# SA_Global
## This code runs the model with few factors parameters combination

## Imports and files

In [1]:
import os
import pickle
import scipy.io as sio
import itertools
import re

from datetime import datetime
from datetime import timedelta

import numpy as np
import pandas as pd
import geopandas as gpd

import contextily as ctx
from shapely.geometry import Polygon
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from datetime import datetime
from collections import namedtuple

from swmm_api.input_file import read_inp_file, SwmmInput, section_labels as sections
from swmm_api import read_inp_file, read_out_file, swmm5_run

## Functions

In [2]:
def get_poly_rain(x, y, radar_gdf):
    """
    Calculates the rain value of the grid point

    Parameters:
    - x: float or int, the x-coordinate of the point of interest
    - y: float or int, the y-coordinate of the point of interest
    - radar_gdf: GeoDataFrame, a GeoDataFrame object containing radar data with rain values and corresponding geometry information

    Return:
    - poly_rain: float, the average rain value of the four grid points within the square polygon centered around the point of interest

    Notes:
    - The point of interest should be within the extent of the GeoDataFrame object.
    - The function assumes that the GeoDataFrame object has a "rain" column and a "geometry" column containing Point objects.
    """

    # create a Point object from the (x, y) coordinates
    cord1= Point(x, y)

    # use .contains() to identify the row(s) that contain the point of interest
    cord_row1 = radar_gdf["geometry"].contains(cord1)

    # extract the rain value from the corresponding row
    rain_value1 = radar_gdf.loc[cord_row1, "rain"].iloc[0]

    # Calculate the average of 4 grid points
    poly_rain = rain_value1
    return poly_rain

In [3]:
def calculate_rain_data(trans_rain_array, BIAS, radar_basin_legend_gdf, time_vector):
    """
    Calculate rainfall data and basin rain information.

    Parameters:
    - trans_rain_array : numpy.ndarray, the transformed rainfall data.
    - BIAS : float, the bias correction factor.
    - time_vector : list, time vector corresponding to the time steps.

    Returns:
    - radar_poly_grid_gdf : geopandas.GeoDataFrame, GeoDataFrame containing rainfall data for each radar cell.
    - basins_rain_df : pandas.DataFrame, DataFrame containing basin rain information.
    """
    # Define geo parameters
    crs = 'epsg:2039'
    length = 500
    wide = 500
    xll = 185500
    yll = 671500
    cellsize = 500

    # Initialize rain_array_3d
    rain_array_3d = trans_rain_array / (20 * BIAS)
    time_steps = rain_array_3d.shape[-1]

    # Initialize rain_data list and polygons
    rain_data = []
    polygons = []

    # Initialize radar_poly_grid_gdf
    radar_poly_grid_gdf = gpd.GeoDataFrame({'geometry': polygons}, crs=crs)

    # Iterate over time steps
    for time in range(time_steps):
        rain_array_2d = rain_array_3d[:, :, time]

        # Create 1D arrays of the X and Y coordinates
        y = np.arange(yll, yll + cellsize * rain_array_2d.shape[0], cellsize)
        x = np.arange(xll, xll + cellsize * rain_array_2d.shape[1], cellsize)

        # Create a 2D grid of X and Y coordinates
        xx, yy = np.meshgrid(x, y)

        # Reshape the arrays into 1D arrays
        x_flat = xx.ravel()
        y_flat = yy.ravel()
        rain_flat = rain_array_2d.flatten()

        # Create a pandas DataFrame with the X and Y coordinates and the rain values
        radar_df = pd.DataFrame({'x': x_flat, 'y': y_flat, 'rain': rain_flat})

        # Create a GeoDataFrame with point geometry
        geometry = gpd.points_from_xy(radar_df.x, radar_df.y)
        radar_gdf = gpd.GeoDataFrame(radar_df, crs=crs, geometry=geometry)

        # Get total bounds of the GeoDataFrame
        xmin, ymin, xmax, ymax = radar_gdf.total_bounds

        # Define the grid Polygon
        cols = list(reversed(np.arange(xmin, xmax + wide, wide)))
        rows = list(reversed(np.arange(ymin, ymax + length, length)))
        rain_l = []

        for y_coord in rows[:-1]:
            for x_coord in cols[:-1]:
                # Get rain data from grid points to polygon
                rain_value = get_poly_rain(x_coord, y_coord, radar_gdf)
                rain_l.append(rain_value)

        rain_data.append({time_vector[time]: rain_l})
        rain_df = pd.DataFrame(rain_data[time])
        radar_poly_grid_gdf = pd.concat([radar_poly_grid_gdf, rain_df], axis=1)

    radar_poly_grid_gdf.index.name = 'radar_cell_num'

    # Sort and re-index radar_basin_legend_gdf
    radar_basin_legend_gdf = radar_basin_legend_gdf.sort_values(by=['Basin_name', 'radar_cell_num']).reset_index(drop=True)

    # Define columns of the final DataFrame
    basin_rain_columns = ['Basin_name'] + list(radar_poly_grid_gdf.columns[1:])
    basins_rain_df = pd.DataFrame()

    for basin in range(1, len(raanana_basins_gdf) + 1):
        # Create DataFrame with pct, relevant radar cell num, and rain timeseries for each basin
        basin_df = radar_basin_legend_gdf[radar_basin_legend_gdf['Basin_name'] == basin][['Basin_name', 'pct', 'radar_cell_num']]
        basin_radar_cells_df = radar_poly_grid_gdf.loc[basin_df['radar_cell_num']] 
        basin_radar_cells_df = pd.merge(basin_df, basin_radar_cells_df, on='radar_cell_num')

        weighted_avgs = []
        for col in basin_rain_columns[1:]:
            weighted_avg = sum(basin_radar_cells_df[col] * basin_radar_cells_df['pct']) 
            weighted_avgs.append(weighted_avg)

        new_row = [basin] + weighted_avgs
        df_basin = pd.DataFrame([new_row], columns=basin_rain_columns)
        basins_rain_df = pd.concat([basins_rain_df, df_basin])

    basins_rain_df = basins_rain_df.reset_index(drop=True)
    basins_rain_df.fillna(0,inplace=True)
    
    return basins_rain_df


In [4]:
def create_TimeSeriesData_txt(basins_rain_df):
    """
    Update the TimeSeriesData values of SWMM inp file by using basins_rain_df

    Args:
        basins_rain_df (DataFrame): DataFrame where each row represents a basin, and each column represents a timestep. The values are rain (mm) for each basin in a timestep.
    Returns:
        str: Text representation of the updated timeseries data in the specified format.
    """
    # Initialize an empty string to store the text
    
    timeseries_text = ''
    # Write the header
    timeseries_text += ";;Name                 Date          Time         Value     \n"
    timeseries_text += ";;------------ ----------------- ------------- -------------\n"

    # Iterate over each row in the DataFrame
    for index, row in basins_rain_df.iterrows():
        # Get the basin name
        basin_name = int(row['Basin_name'])

        # Write the data to the text
        timestamps = basins_rain_df.columns[1:]  # excluding the first column 'Basin_name'
        values = row.values[1:]  # excluding the first column 'Basin_name'
        dates = [ts.strftime('%m/%d/%Y') for ts in timestamps]  # convert date format
        times = [ts.strftime('%H:%M:%S') for ts in timestamps]  # convert time format
        for i in range(len(timestamps)):
            timeseries_text += f"  {date}_S{basin_name}      {dates[i]}    {times[i]}   {values[i]:.2f}\n"
        timeseries_text += ";;------------ ----------------- ------------- -------------\n"
    return timeseries_text


In [5]:
def BoxCoxTransform(rain_array, lam):
    """
    Apply Box-Cox transformation to rainfall data.
    Parameters:
    -----------
    rain_array : array_like -> Input array containing rainfall data.    
    lam : float -> Lambda parameter for transformation.
    Returns:
    --------
    trans_rain_array : numpy.ndarray -> Transformed rainfall data.    
    bias : float -> Bias factor for normalization.
    """
    
    # change nans values to 0
    rain_array = np.nan_to_num(rain_array, nan=0)
    # Perform Box-Cox transformation
    trans_rain_array = np.where(((rain_array**lam)-1) > 0, (((rain_array**lam)-1)/lam), 0)

    transformation_bias = trans_rain_array.sum() / rain_array.sum()

    # Normalize data to maintain total storm value
    trans_rain_array = trans_rain_array / transformation_bias
    
    return trans_rain_array, transformation_bias


In [6]:
def update_TimeSeriesData(SIM_DIR, date, transform_timeseries_text):
    """
    Update the time series data in an input file used for simulation.
    Args:
    - SIM_DIR (str): Directory where simulation files are located.
    - date (str): Date string used to locate simulation files.
    - transform_timeseries_text (text): text containing transformed time series data.
    Returns:
    - None
    """
    INP_FILE = r"Final.inp"   ## This is the file name before manipulations

    SIM_PATH = os.path.join(SIM_DIR, date + '/')
    inp = read_inp_file(SIM_PATH + INP_FILE)
    timesereies_d = dict(inp[sections.TIMESERIES])
    inp[sections.TIMESERIES] = transform_timeseries_text
    
        ## make sure that time serie data fit number of 'raingauges'
    if len(inp[sections.TIMESERIES].keys()) != len(inp[sections.RAINGAGES].keys()):
        print('ERROR')

    ## Edit the time series name and its corresponding rain gauge field so that they will be the same
    for basin_timeserie in range(len(list(inp[sections.TIMESERIES].keys()))):
    #     print(basin_timeserie)
        basin_timeserie_name = inp[sections.TIMESERIES][list(inp[sections.TIMESERIES].keys())[basin_timeserie]]['name']
        inp[sections.RAINGAGES][list(inp[sections.RAINGAGES].keys())[basin_timeserie]]['timeseries'] = basin_timeserie_name

    file_name = 'raanana_sensitivity_analysis'
    inp.write_file(SIM_PATH + file_name + '.inp')


In [7]:
def update_imperviousness_with_factor(subcatchment_dict, factor):
    """
    Update the imperviousness values of SubCatchment objects in a dictionary by a factor.

    Args:
        subcatchment_dict (dict): A dictionary containing SubCatchment objects as values with subcatchment names as keys.
        factor (float): The factor by which imperviousness values need to be powered.

    Returns:
        dict: The updated dictionary with imperviousness values powered by the factor.
    """
#     impervious_initial_values = [21, 43, 47, 47, 51, 47, 51, 47, 43, 22, 37, 18, 24, 37, 47, 35, 45, 71, 19]
    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, subcatchment) in enumerate(subcatchment_dict.items()):
#         initial_imperviousness = impervious_initial_values[index]
#         subcatchment.imperviousness = initial_imperviousness
        powered_imperviousness = subcatchment.imperviousness * factor
        # Update the imperviousness value for the current subcatchment with the powered imperviousness value
        subcatchment.imperviousness = powered_imperviousness


    # Return the updated dictionary
    return subcatchment_dict


def update_storage_with_factor(subareas_dict, factor):
    """
    Update the storage values of SubArea objects in a dictionary by a factor.

    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        factor (float): The factor by which storage values need to be powered.

    Returns:
        dict: The updated dictionary with storage values powered by the factor.
    """
    # Loop through each subarea in the dictionary
    for subarea_name, subarea in subareas_dict.items():
        # Access the impervious and pervious storage values for the current subarea
        imp_storage = subarea.storage_imperv
        perv_storage = subarea.storage_perv
        # Multiply the impervious and pervious storage values by the factor
        powered_imp_storage = imp_storage * factor
        powered_perv_storage = perv_storage * factor
        # Update the storage values for the current subarea with the powered values
        subarea.storage_imperv = powered_imp_storage
        subarea.storage_perv = powered_perv_storage

    # Return the updated dictionary
    return subareas_dict

def update_n_with_factor(subareas_dict, factor):
    """
    Update the n_imperv and n_perv values of SubArea objects in a dictionary by a factor.

    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        factor (float): The factor by which n_imperv and n_perv values need to be multiplied.

    Returns:
        dict: The updated dictionary with n_imperv and n_perv values multiplied by the factor.
    """
    # Loop through each subarea in the dictionary
    for subarea_name, subarea in subareas_dict.items():
        # Access the impervious and pervious n values for the current subarea
        imp_n = subarea.n_imperv
        perv_n = subarea.n_perv
        # Multiply the impervious and pervious n values by the factor
        updated_imp_n = imp_n * factor
        updated_perv_n = perv_n * factor
        # Update the n values for the current subarea with the multiplied values
        subarea.n_imperv = updated_imp_n
        subarea.n_perv = updated_perv_n

    # Return the updated dictionary
    return subareas_dict


def update_width_with_factor(subcatchment_dict, factor, width_initial_values):
    """
    Update the width values of SubCatchment objects in a dictionary by a factor.

    Args:
        subcatchment_dict (dict): A dictionary containing SubCatchment objects as values with subcatchment names as keys.
        factor (float): The factor by which width values need to be powered.
        width_initial_values (numpy.ndarray): A NumPy array of initial width values corresponding to each subcatchment.

    Returns:
        dict: The updated dictionary with width values powered by the factor.
    """
    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, subcatchment) in enumerate(subcatchment_dict.items()):
        width = width_initial_values[index]  # Get the initial width value for the current subcatchment
        powered_width = width * factor
        subcatchment.width = powered_width

    # Return the updated dictionary
    return subcatchment_dict


def update_curve_num_with_factor(infiltration_dict, cn_factor):
    """
    Update the curve_no values of InfiltrationCurveNumber objects in a dictionary by a factor.

    Args:
        infiltration_dict (dict): A dictionary containing InfiltrationCurveNumber objects as values with subcatchment
                                  names as keys.
        cn_factor (float): The factor by which curve_no values need to be multiplied.

    Returns:
        dict: The updated dictionary with curve_no values multiplied by the factor.
    """
    # Loop through each subcatchment in the dictionary
    for subcatchment_name, infiltration_curve in infiltration_dict.items():
        # Get the initial curve_no value for the current subcatchment
        curve_no = int(infiltration_curve.curve_no)
        # Multiply the curve_no value by the factor
        powered_curve_no = curve_no * cn_factor
        # Update the curve_no value for the current subcatchment with the multiplied value
        infiltration_curve.curve_no = powered_curve_no

    # Return the updated dictionary
    return infiltration_dict

def update_pct_zero_with_factor(subareas_dict, pct_zero_factor):
    """
    Update the pct_zero values of SubArea objects in a dictionary by a factor.
    
    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        pct_zero_factor (float): The factor by which pct_zero values need to be multiplied.
        pct_zero_initial_values (numpy.ndarray): A NumPy array of initial pct_zero values corresponding to each subarea.
    
    Returns:
        dict: The updated dictionary with pct_zero values multiplied by the factor.
    """
    # Loop through each subarea in the dictionary
    for index, (subarea_name, subarea) in enumerate(subareas_dict.items()):
        # Get the initial pct_zero value for the current subarea
        pct_zero = subarea.pct_zero
        # Multiply the pct_zero value by the factor
        updated_pct_zero = pct_zero * pct_zero_factor
        # Update the pct_zero value for the current subarea with the multiplied value
        subarea.pct_zero = updated_pct_zero
    
    # Return the updated dictionary
    return subareas_dict

In [8]:
def load_runoff_obs_to_df(OBS_RUNOFF_DATA_PATH):
    """ Takes OBSERVATION FILE with runoff data and makes pd Dataframe
    :param output_path: str, path name that have the runoff data
    :return: df_obs_runoff
    """
    df_obs_runoff = pd.read_csv (OBS_RUNOFF_DATA_PATH + ".csv")
    df_obs_runoff.dropna(inplace = True) ; df_obs_runoff.rename(columns={'discharge_cms': 'OBS runoff [CMS]'} , inplace=True, errors='raise')
    df_obs_runoff["date_and_time"] =  pd.to_datetime(df_obs_runoff["date_and_time"], format='%d/%m/%Y %H:%M:%S')
    df_obs_runoff = df_obs_runoff.set_index('date_and_time')
    df_obs_runoff.drop(['Unnamed: 0'], axis=1, inplace=True)


    return df_obs_runoff
    
# df_runoff.to_pickle('df_runoff.pkl')

In [9]:
def data_5min_interpolate(df_to_interpolate):
    """ Takes df column and makes 5 min interpolation
    :param df_to_interpolate: pd, df with index datetime64 type
    :return: df_interpol
    """
    df_interpol = df_to_interpolate.resample('5min').mean()
#     df_interpol= df_interpol.interpolate(method='polynomial', order=3)
    df_interpol= df_interpol.interpolate(method='linear')
    return df_interpol

In [10]:
def load_sim_to_df(sim_output_path):
    """ Takes SWMM FILE with runoff data and makes pd Dataframe
    sim_output_path: str, file name that have the swmm runoff data
    return: sim_df
    """
    sim_df = read_out_file(sim_output_path).to_frame()['system'][''][['outflow', 'rainfall']]
    sim_df.rename(columns={'outflow':'SWMM outflow [CMS]', 'rainfall':'rainfall [mm/h]'}, inplace=True, errors='raise')
    sim_df.index = pd.to_datetime(sim_df.index)
    return sim_df

In [11]:
def observation_and_swmm_hydrograph(storm_df, date):
    """
    This function takes observation and SWMM model runoff and precipitation data and puts it in a hydrograph.
    Params:
     - storm_df: df, The DataFrame that includes the runoff and precipitation data
     - date: date of storm occurrence. Format: 'yyyy_mm_dd'
    Returns: a hydrograph plot for one subcatchment
    """
    
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.suptitle(storm_df.index.strftime('%d/%m/%Y')[0], fontsize=20)

    y_obs_runoff = storm_df['OBS runoff [CMS]']
    y_rainfall = storm_df['rainfall [mm/h]']
    y_sim_runoff = storm_df['SWMM outflow [CMS]']

    ax1 = sns.lineplot(ax=ax, data=y_obs_runoff, color='g', label='Observed Runoff')
    ax2 = ax1.twinx()
#     ax2.set_ylabel('Rainfall ($mm/hr$)')

    sns.lineplot(ax=ax2, data=y_rainfall, color='b', label='Rainfall', alpha=0.4)
    ax2.fill_between(storm_df.index, 0, y_rainfall, alpha=0.4, color='b')
#     ax2.set_ylim(ymin=y_rainfall.max() + 50, ymax=0)
    ax2.set_ylim(ymin=70, ymax=0)
    ax1.set_ylim(ymin=0, ymax=90)
    

    sns.lineplot(ax=ax, data=y_sim_runoff, color='r', label='Simulated Runoff')

    ax2.set_xlabel('Time', fontsize=18)
    ax2.set_ylabel('Rainfall ($mmhr^{-1}$)', fontsize=18)
    ax1.set_ylabel('Runoff ($m^{3}s^{-1}$)', fontsize=18)

    ax2.legend().remove()

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    handles = handles1 + handles2
    labels = labels1 + labels2
    ax.legend(handles, labels, loc='center right', fontsize=15)

    # set x-axis label format to %d/%m %H
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %H:%M'))
    ax.tick_params(axis='x', labelsize=18, rotation=45)
    ax1.tick_params(axis='y', labelsize=18) ; ax2.tick_params(axis='y', labelsize=18)
    ax1.grid()


    return fig


In [12]:
def stat(storm_df):
    """
    Format statistical data from storm_df DataFrame.
    
    Calculates various statistical parameters based on the storm data in the storm_df DataFrame.
    
    Args:
        storm_df (pandas.DataFrame): DataFrame containing storm data from SWMM and OBS.
        
    Returns:
        Tuple containing the following:
        
        - swmm_total_runoff (float): Total runoff volume in cubic meters calculated from SWMM data.
        - obs_total_runoff (float): Total runoff volume in cubic meters calculated from OBS data.
        - swmm_max_runoff (float): Runoff peak in CMS (Cubic Meters per Second) from SWMM data.
        - obs_max_runoff (float): Runoff peak in CMS (Cubic Meters per Second) from OBS data.
        - swmm_max_runoff_time (datetime): Time of the runoff peak from SWMM data.
    """
    
    # Total runoff volume [cubic meter]
    swmm_total_runoff = sum(storm_df['SWMM outflow [CMS]'])
    obs_total_runoff = sum(storm_df['OBS runoff [CMS]'])

    # Runoff peak [CMS] and time
    swmm_max_runoff = storm_df['SWMM outflow [CMS]'].max()
    swmm_max_runoff_time = storm_df[storm_df['SWMM outflow [CMS]'] == swmm_max_runoff].index[0]

    obs_max_runoff = storm_df['OBS runoff [CMS]'].max()
    obs_max_runoff_time = storm_df[storm_df['OBS runoff [CMS]'] == obs_max_runoff].index[0]

    return swmm_total_runoff, obs_total_runoff, swmm_max_runoff, obs_max_runoff


## Load Data

In [13]:
# Load radar_basin_legend_gdf
directory = r"D:\Development\RESEARCH\Raanana\data\rain_radar\Basin_radar_overlap_pkl"
filename = "radar_basin_legend_gdf.pkl"
filepath = os.path.join(directory, filename)

with open(filepath, "rb") as f:
    radar_basin_legend_gdf = pickle.load(f)


In [14]:
# Load the shapefiles for Raanana
raanana_basins_shapfile = r'D:\Development\RESEARCH\Raanana\gis\GIS\28_subcatchments\raanana_28_subcatchments.shp'
raanana_basins_gdf = gpd.read_file(raanana_basins_shapfile)  # sub-basins poly
raanana_basins_gdf.drop(columns=['Shape_Area', 'Area_km2', 'Area_ha','Area_m2'], inplace = True)


# Rain-Radar Transformation
## fromgrided matfile through transform basins rain txt file to inf file timeseries

In [15]:
# Load the storm MAT file
radar_dir = r'D:\Development\RESEARCH\Raanana\data\rain_radar\Row_radar_correct_MAT_files/'

## SELECT the storm event:
filename = '20191213'
rain_gauge_bias = 0.87
radar_data = sio.loadmat(radar_dir + filename)
date = '_'.join([filename[:4], filename[4:6], filename[6:]])

# Rain Data
rain_array_3d= radar_data['event_file'][0][0][0]
# Time Data 
## The Time 1d values 
time_vector= radar_data['event_file'][0][0][1].ravel()
time_vector = pd.to_datetime(time_vector-719529, unit='D').round('1min') # Convert the time from float to datetime and round 


 # Run the simulation

In [16]:
## Find the original Z of the calibrated model
sim_dir = r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/"
sim_dir = os.path.join(sim_dir, date + '/')
INP_FILE = r"Final.inp"

OBS_FILE = date 
obs_runoff_df = data_5min_interpolate(load_runoff_obs_to_df(sim_dir + OBS_FILE))

inp = read_inp_file(sim_dir + INP_FILE) 
file_name = 'find_z'

inp.write_file(sim_dir + file_name + '.inp')

swmm5_run(sim_dir + file_name + '.inp', progress_size=1)
OUT_FILE = file_name +'.out'

sim_df = load_sim_to_df(sim_dir + OUT_FILE)

# Concat observed and simulated data
storm_df = pd.concat([obs_runoff_df, sim_df], axis=1)
storm_df[storm_df < 0] = 0
storm_df = storm_df.fillna(0)


# Calculate and store important values
original_total_runoff, obs_total_runoff, original_max_runoff, obs_max_runoff = stat(storm_df)
original_max_rainfall = storm_df['rainfall [mm/h]'].max()

original_total_runoff =  original_total_runoff*60*5   ## from m3/s to m3

print('max rain intense:' , original_max_rainfall)
print('date:' , date)
print('swmm total runoff:' , original_total_runoff)
print('swmm max runoff:' , original_max_runoff)

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/find_z.inp:   0%|       …

max rain intense: 33.30265426635742
date: 2019_12_13
swmm total runoff: 86353.38953726081
swmm max runoff: 23.523775100708008


# GLOBAL SA

In [17]:
# lam_factors = np.arange(0.5, 1.1, 0.5)  # Values from 0.4 to 1.2 with 0.05 intervals
# imp_factors = np.arange(0.5, 1.1, 0.5) # cant be bigger then 1.39
# n_factors = np.arange(0.5, 1.1, 0.5)
# pct_zero_factors = np.arange(0.5, 1.1, 0.5) # cant be bigger then 1.44
# cn_factors = np.arange(0.5, 1.1, 0.5)   # cant be bigger then 2.56
# storage_factors = np.arange(0.5, 1.1, 0.5)

# # storage_factors_1 = np.arange(0.2, 1.1, 0.4)
# # storage_factors_2 = np.arange(2, 8, 2)

# # Concatenate arrays to get the final storage_factors array
# # storage_factors = np.concatenate([storage_factors_1, storage_factors_2])
# FACTOR_COMBINATION = np.array(list(itertools.product(lam_factors, imp_factors, storage_factors, n_factors, cn_factors, pct_zero_factors)))
# len(FACTOR_COMBINATION)


# # Split into 10 runs
# num_runs = 8

# chunk_size = len(FACTOR_COMBINATION) // num_runs

# # Divide the array into chunks
# factor_combinations_runs_array = np.array(np.split(FACTOR_COMBINATION, num_runs))

# # Now, factor_combinations_runs_array is a NumPy array containing 2 subarrays, each representing a portion of the combinations.
# print(f'Chunk Size {chunk_size}')


In [18]:

def read_coxbox_txt_file(coxbox_directory_path, lam_factor):
    # List all files in the directory
    files_in_directory = os.listdir(coxbox_directory_path)

    # Filter files with the desired format
    matching_files = [file for file in files_in_directory if "transform_timeseries_data_lam_" in file]

    if not matching_files:
        print("No matching files found.")
        return None, None

    # Find the file that matches the specified lam_factor
    lam_factor_str = str(lam_factor)
    matching_file = None
    for file in matching_files:
        if re.search(r'lam_([\d.]+)\.', file):
            file_lam_factor = float(re.search(r'lam_([\d.]+)\.', file).group(1))
            if file_lam_factor == lam_factor:
                matching_file = file
                break

    if not matching_file:
        print(f"No file found with lam_factor: {lam_factor}")
        return None, None

    # Construct the file path
    file_path = os.path.join(coxbox_directory_path, matching_file)

    # Open and read the contents of the matched file
    with open(file_path, 'r') as file:
        coxbox_txt = file.read()

    # Return lam_factor and file_contents
    return lam_factor, coxbox_txt

# Example usage
# lam_factor, coxbox_txt = read_specific_file('path_to_directory', 1.5


In [19]:
def update_TimeSeriesData(inp, timeseries_text, output_file_name):
    """
    Update the time series data in an input file used for simulation.
    Args:
    - SIM_DIR (str): Directory where simulation files are located.
    - date (str): Date string used to locate simulation files.
    - timeseries_text (text): text containing transformed time series data.
    Returns:
    - None
    """
#     INP_FILE = r"Final.inp"   ## This is the file name before manipulations

#     SIM_PATH = os.path.join(SIM_DIR, date + '/')
#     inp = read_inp_file(SIM_PATH + INP_FILE)
#     timesereies_d = dict(inp[sections.TIMESERIES])
    inp[sections.TIMESERIES] = timeseries_text
    
        ## make sure that time serie data fit number of 'raingauges'
    if len(inp[sections.TIMESERIES].keys()) != len(inp[sections.RAINGAGES].keys()):
        print('ERROR')

    ## Edit the time series name and its corresponding rain gauge field so that they will be the same
    for basin_timeserie in range(len(list(inp[sections.TIMESERIES].keys()))):
    #     print(basin_timeserie)
        basin_timeserie_name = inp[sections.TIMESERIES][list(inp[sections.TIMESERIES].keys())[basin_timeserie]]['name']
        inp[sections.RAINGAGES][list(inp[sections.RAINGAGES].keys())[basin_timeserie]]['timeseries'] = basin_timeserie_name

#     file_name = 'raanana_sensitivity_analysis'
#     inp.write_file(SIM_PATH + output_file_name + '.inp')


In [20]:
lam_factors = np.around(np.arange(0.2, 2, 0.4), decimals=2)
imp_factors = np.arange(0.1, 1.5, 0.15) # cant be bigger then 1.39
n_factors = np.arange(0.5, 3, 0.5)
cn_factors = (0.5, 2.5, 0.5)   # cant be bigger then 2.56
pct_zero_factors = (0.1, 1.4, 0.3) # cant be bigger then 1.44

storage_factors_1 = np.arange(0.2, 1.1, 0.4)
storage_factors_2 = np.arange(2, 8, 2)

# Concatenate arrays to get the final storage_factors array
storage_factors = np.concatenate([storage_factors_1, storage_factors_2])
FACTOR_COMBINATION = np.array(list(itertools.product(lam_factors, imp_factors, storage_factors, n_factors, cn_factors, pct_zero_factors)))
len(FACTOR_COMBINATION)


# Split into 10 runs
num_runs = 50
chunk_size = len(FACTOR_COMBINATION) // num_runs

# Divide the array into chunks
factor_combinations_runs_array = np.array(np.split(FACTOR_COMBINATION, num_runs))

# Now, factor_combinations_runs_array is a NumPy array containing 2 subarrays, each representing a portion of the combinations.
print(f'Chunk Size {chunk_size}')



Chunk Size 270


In [35]:
pickle_dir = os.path.join(sim_dir,'pickles' + '/')

for chunk_num, chunk_factor_combination in enumerate(factor_combinations_runs_array):
    lam_factors_l = []
    imp_factor_l = []
    storage_factor_l = []
    n_factor_l = []
    cn_factor_l = []
    pct_zero_factor_l = []
    max_rain_values_l = []
    swmm_total_runoff_l = []
    swmm_max_runoff_l = []

    for lam_factor, imp_factor, storage_factor, n_factor, cn_factor, pct_zero_factor in chunk_factor_combination:

        sim_dir = r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/"
        sim_dir = os.path.join(sim_dir, date + '/')

        INP_FILE = r"Final.inp"
        output_file_name = 'raanana_Global_withcoxbox_sensitivity_analysis'
                
        inp = read_inp_file(os.path.join(sim_dir, INP_FILE))

        # Select the relevant sections
        subcatchment_d = dict(inp[sections.SUBCATCHMENTS])
        infiltration_d = dict(inp[sections.INFILTRATION])
        subareas_d = dict(inp[sections.SUBAREAS])
        timeseries_d = dict(inp[sections.TIMESERIES])
        raingauge_d = dict(inp[sections.RAINGAGES])

        # Update the section value by factor
        update_imperviousness_with_factor(subcatchment_d, imp_factor)
        update_storage_with_factor(subareas_d, storage_factor)
        update_n_with_factor(subareas_d, n_factor)        
        update_pct_zero_with_factor(subareas_d, pct_zero_factor)
        update_curve_num_with_factor(infiltration_d, cn_factor)
                        
#         coxbox_directory_path = r"D:\Development\RESEARCH\Raanana\data\rain_radar\SA_rain_radar_transformation\CoxBox"


        coxbox_directory_path = f"D:\\Development\\RESEARCH\\Raanana\\data\\rain_radar\\SA_rain_radar_transformation\\{date.replace('_', '')}\CoxBox"

        lam_factor, coxbox_timeserie_txt = read_coxbox_txt_file(coxbox_directory_path, lam_factor)
        update_TimeSeriesData(inp, coxbox_timeserie_txt, output_file_name)
        
        inp.write_file(os.path.join(sim_dir, output_file_name + '.inp'))
        swmm5_run(os.path.join(sim_dir, output_file_name + '.inp'), progress_size=1)
        OUT_FILE = output_file_name + '.out'

        sim_df = load_sim_to_df(os.path.join(sim_dir, OUT_FILE))
        storm_df = pd.concat([obs_runoff_df, sim_df], axis=1)
        storm_df[storm_df < 0] = 0
        storm_df = storm_df.fillna(0)

    #     plt.plot(storm_df.index, storm_df['rainfall [mm/h]'], label=f'lam={lam}')

        swmm_total_runoff, obs_total_runoff, swmm_max_runoff, obs_max_runoff = stat(storm_df)
        swmm_total_runoff_l.append(swmm_total_runoff*60*5)    ## fix it
        swmm_max_runoff_l.append(swmm_max_runoff)
        max_rain_values_l.append(storm_df['rainfall [mm/h]'].max())

        lam_factors_l.append(lam_factor)
        imp_factor_l.append(imp_factor)
        storage_factor_l.append(storage_factor)
        n_factor_l.append(n_factor)
        cn_factor_l.append(cn_factor)
        pct_zero_factor_l.append(pct_zero_factor)
        
    # Create DataFrame
    global_sa_df = pd.DataFrame({
        'lam_factor': lam_factors_l,
        'imp_change': [(x - 1) * 100 for x in imp_factor_l],
        'storage_change': [(x - 1) * 100 for x in storage_factor_l],
        'n_change':[(x - 1) * 100 for x in n_factor_l],
        'cn_change':[(x - 1) * 100 for x in cn_factor_l],
        'pct_zero_change':[(x - 1) * 100 for x in pct_zero_factor_l],
        'max_rain_val': max_rain_values_l,
        'swmm_max_runoff': swmm_max_runoff_l,
        'swmm_total_runoff': swmm_total_runoff_l,
    })

    # Calculate percentage change for each parameter
    global_sa_df['max_rainfall_change'] = ((global_sa_df['max_rain_val'] / original_max_rainfall) - 1) * 100
    global_sa_df['max_runoff_change_percent'] = ((global_sa_df['swmm_max_runoff'] / original_max_runoff) - 1) * 100
    global_sa_df['total_runoff_change_percent'] = ((global_sa_df['swmm_total_runoff'] / original_total_runoff) - 1) * 100

    # Reorder global_sa_df
    global_sa_df = global_sa_df[['max_rainfall_change', 'imp_change', 'storage_change', 'n_change', 'cn_change',  'pct_zero_change', 'total_runoff_change_percent',
                              'swmm_total_runoff','swmm_max_runoff']]
    # save to pickle:
    global_sa_df.to_pickle(os.path.join(pickle_dir, f"global_sa_pickle{chunk_num}.pkl"))


swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

...

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/2019_12_13/raanana_Global_withcoxbo…

KeyboardInterrupt: 

In [ ]:
global_sa_df